## Module 5(1) - Classification Modeling (Tree-based)
Asst. Prof. Dr. Santitham Prom-on

---

# 1. Load data

In [ ]:
import pandas as pd
pd.set_option('display.max_columns',100)
df = pd.read_csv('hr_attrition.csv')
df.head()

In [ ]:
df['BusinessTravel'].value_counts()

In [ ]:
df.info()

# 2. Explore and clean data (data wrangling)

## Numeric columns

In [ ]:
df_num = df.select_dtypes('number')
df_num.head()

In [ ]:
df_num.describe().T

In [ ]:
df_num.drop(columns=['EmployeeCount','EmployeeNumber','StandardHours'], inplace=True)

In [ ]:
df_num.columns

## Categorical columns

In [ ]:
df_cat = df.select_dtypes('object')
df_cat.head()

In [ ]:
df_cat.describe().T

In [ ]:
df_cat.drop(columns='Over18', inplace=True)

In [ ]:
df_cat['Attrition'].value_counts()

In [ ]:
df_cat['BusinessTravel'].value_counts()

In [ ]:
df_cat['Department'].value_counts()

In [ ]:
df_cat['EducationField'].value_counts()

In [ ]:
df_cat.replace({'EducationField':{'Microbiology':'Life Sciences', 'Marketing and Advertisement':'Marketing'}}, inplace=True)
df_cat['EducationField'].value_counts()

In [ ]:
df_cat['Gender'].value_counts()

In [ ]:
df_cat['JobRole'].value_counts()

In [ ]:
df_cat['MaritalStatus'].value_counts()

In [ ]:
df_cat['OverTime'].value_counts()

In [ ]:
df_cat.head()

In [ ]:
pd.get_dummies(df_cat['BusinessTravel'])

In [ ]:
pd.get_dummies(df_cat['BusinessTravel'], drop_first=True)

In [ ]:
df_cat_bin = pd.get_dummies(df_cat, drop_first=True)
df_cat_bin.head()

## Combine both numerical and categorical data

In [ ]:
df_prep = pd.concat([df_cat_bin, df_num], axis=1)

## Holdout

In [ ]:
X = df_prep.drop(columns='Attrition_Yes')
y = df_prep['Attrition_Yes']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=777)

In [ ]:
y_train.value_counts()/y_train.shape[0]

In [ ]:
y_test.value_counts()/y_test.shape[0]

# Save data for the next module

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pickle

pickle.dump((X_train,X_test,y_train,y_test),
            open('/content/drive/MyDrive/GSB/hr_attrition.data','wb'))

# Modeling

In [ ]:
X_train.shape

In [ ]:
X_train.head()

In [ ]:
%%time
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(min_samples_leaf=20, max_depth=8, class_weight='balanced')
#tree = DecisionTreeClassifier(max_depth=8, class_weight='balanced')
tree.fit(X_train,y_train)

# 5. Plot a decision tree

In [ ]:
from six import StringIO
from IPython.display import Image
from sklearn.tree import export_graphviz
import pydotplus
dot_data = StringIO()
export_graphviz(tree, out_file=dot_data,
                filled=True, rounded=True,
                special_characters=True,
                rotate=True,
                feature_names=X_train.columns)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
graph.write_png('tree.png')
Image(graph.create_png())

# 6. Variable importance

In [ ]:
tree.feature_importances_

In [ ]:
pd.DataFrame(dict(Feature=tree.feature_names_in_,
                  Value=tree.feature_importances_))

In [ ]:
pd.DataFrame(dict(Feature=tree.feature_names_in_,
                  Value=tree.feature_importances_))\
  .sort_values(by='Value', ascending=False)\
  .head(20)

# 7. Prediction

In [ ]:
y_pred_class = tree.predict(X_test)
y_pred_class
# threshold prob = 0.5
# max score

In [ ]:
y_pred_prob = tree.predict_proba(X_test)
y_pred_prob

In [ ]:
y_pred_prob[:10,:]

# 8. Confusion matrix and classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
confusion_matrix(y_true = y_test, y_pred = y_pred_class)

In [ ]:
print(classification_report(y_true= y_test, y_pred = y_pred_class))

# 9. AUC ROC analysis

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:,1])
roc_auc = auc(fpr, tpr)
display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc,
                                  estimator_name='Decision Tree')
display.plot()